# Revisão manual de temas — taxonomia, macrotema e subtema

Este notebook prepara, classifica e salva um **dicionário manual de temas** para ser reaplicado no pipeline.

A lógica é:

1. Ler `temas_normalizados.csv`.
2. Criar uma base única por `tema_normalizado`.
3. Aplicar regras automáticas por palavras-chave.
4. Revisar pendências manualmente.
5. Salvar:
   - `dicionario_temas.csv`
   - `dim_tema_curada.csv`
   - `pendencias_temas.csv`

> Regra de ouro: não edite o CSV final do BI. Edite o dicionário e reaplique.

In [1]:
from pathlib import Path
import re
import ast
import json
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

## 1. Configuração de caminhos

O código abaixo tenta localizar automaticamente a raiz do projeto, mesmo que o notebook esteja dentro da pasta `notebooks/`.

In [ ]:
from pathlib import Path

def get_project_root():
    path = Path.cwd()

    # sobe até encontrar a pasta "data"
    while path != path.parent:
        if (path / "data").exists():
            return path
        path = path.parent

    raise Exception("Não foi possível encontrar a raiz do projeto (pasta com /data)")

ROOT = get_project_root()

DIR_03 = ROOT / "data" / "03_transformed"
DIR_REVIEW = ROOT / "data" / "04_review"
DIR_ENTRADA = DIR_REVIEW / "entrada_para_revisao"
DIR_DICIONARIOS = DIR_REVIEW / "dicionarios_manuais"
DIR_SAIDA = DIR_REVIEW / "revisoes_aplicadas"
DIR_LOGS = DIR_REVIEW / "logs_revisao"

for pasta in [DIR_ENTRADA, DIR_DICIONARIOS, DIR_SAIDA, DIR_LOGS]:
    pasta.mkdir(parents=True, exist_ok=True)

# Caminhos prováveis para o arquivo de temas
POSSIVEIS_TEMAS = [
    DIR_03 / "tabelas_normalizadas" / "temas_normalizados.csv",
    DIR_03 / "normalizacao_ia" / "bases_normalizadas" / "temas_normalizados.csv",
    DIR_03 / "valores_unicos_para_normalizacao" / "temas_normalizados.csv",
]

ARQ_TEMAS_NORMALIZADOS = next((p for p in POSSIVEIS_TEMAS if p.exists()), None)

# Saídas
ARQ_DICIONARIO_TEMAS = DIR_DICIONARIOS / "dicionario_temas.csv"
ARQ_DIM_TEMA_CURADA = DIR_SAIDA / "dim_tema_curada.csv"
ARQ_PENDENCIAS_TEMAS = DIR_SAIDA / "pendencias_temas.csv"
ARQ_RELATORIO_COBERTURA = DIR_LOGS / "relatorio_cobertura_temas.csv"

print("ROOT:", ROOT)
print("Arquivo de temas encontrado:", ARQ_TEMAS_NORMALIZADOS)
print("Existe?", ARQ_TEMAS_NORMALIZADOS.exists() if ARQ_TEMAS_NORMALIZADOS else False)
print("Dicionário:", ARQ_DICIONARIO_TEMAS)

ROOT: c:\pdf_inventory_reestruturado_v6_final_cache_modelos\notebooks
Arquivo de temas encontrado: None
Existe? False
Dicionário: c:\pdf_inventory_reestruturado_v6_final_cache_modelos\notebooks\data\04_review\dicionarios_manuais\dicionario_temas.csv


## 2. Ler a base de temas normalizados

Se o arquivo não for encontrado, rode a célula seguinte de diagnóstico para localizar onde ele está.

In [4]:
if ARQ_TEMAS_NORMALIZADOS is None:
    print("Arquivo temas_normalizados.csv não encontrado nos caminhos esperados.")
    print("Procurando no projeto...")
    encontrados = list(ROOT.rglob("temas_normalizados.csv"))
    for p in encontrados:
        print(p)
    raise FileNotFoundError("Ajuste ARQ_TEMAS_NORMALIZADOS com um dos caminhos acima.")

temas = pd.read_csv(ARQ_TEMAS_NORMALIZADOS, encoding="utf-8-sig")
print(temas.shape)
display(temas.head())
print(temas.columns.tolist())

Arquivo temas_normalizados.csv não encontrado nos caminhos esperados.
Procurando no projeto...


FileNotFoundError: Ajuste ARQ_TEMAS_NORMALIZADOS com um dos caminhos acima.

## 3. Padronizar colunas

O pipeline pode gerar `tema_original`/`tema_normalizado`, ou nomes próximos. Esta célula tenta padronizar automaticamente.

In [ ]:
def escolher_coluna(df: pd.DataFrame, candidatos: list[str]) -> str | None:
    for col in candidatos:
        if col in df.columns:
            return col
    return None

COL_TEMA_ORIGINAL = escolher_coluna(temas, ["tema_original", "valor_original", "tema", "temas"])
COL_TEMA_NORMALIZADO = escolher_coluna(temas, ["tema_normalizado", "valor_normalizado", "tema_norm", "tema"])

if COL_TEMA_NORMALIZADO is None:
    raise ValueError("Não encontrei coluna de tema normalizado. Colunas disponíveis: " + str(temas.columns.tolist()))

if COL_TEMA_ORIGINAL is None:
    COL_TEMA_ORIGINAL = COL_TEMA_NORMALIZADO

print("COL_TEMA_ORIGINAL:", COL_TEMA_ORIGINAL)
print("COL_TEMA_NORMALIZADO:", COL_TEMA_NORMALIZADO)

## 4. Criar ou carregar o dicionário de revisão

Se `dicionario_temas.csv` já existir, ele será carregado para preservar seu trabalho anterior. Se não existir, será criado a partir de `temas_normalizados.csv`.

In [ ]:
def limpar_texto(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

if ARQ_DICIONARIO_TEMAS.exists():
    print("Dicionário existente encontrado. Carregando para preservar revisões...")
    base_revisao = pd.read_csv(ARQ_DICIONARIO_TEMAS, encoding="utf-8-sig")
else:
    print("Criando novo dicionário de temas...")
    tmp = temas.copy()
    tmp[COL_TEMA_NORMALIZADO] = tmp[COL_TEMA_NORMALIZADO].apply(limpar_texto)
    tmp[COL_TEMA_ORIGINAL] = tmp[COL_TEMA_ORIGINAL].apply(limpar_texto)
    tmp = tmp[tmp[COL_TEMA_NORMALIZADO] != ""]

    variantes = (
        tmp.groupby(COL_TEMA_NORMALIZADO)[COL_TEMA_ORIGINAL]
        .apply(lambda s: " | ".join(sorted(set([v for v in s if v]))))
        .reset_index()
        .rename(columns={COL_TEMA_NORMALIZADO: "tema_normalizado", COL_TEMA_ORIGINAL: "variantes_originais"})
    )

    ocorrencias = (
        tmp.groupby(COL_TEMA_NORMALIZADO)
        .size()
        .reset_index(name="qtd_ocorrencias")
        .rename(columns={COL_TEMA_NORMALIZADO: "tema_normalizado"})
    )

    base_revisao = variantes.merge(ocorrencias, on="tema_normalizado", how="left")
    base_revisao["macrotema"] = ""
    base_revisao["subtema"] = ""
    base_revisao["manter"] = True
    base_revisao["observacao"] = ""

# Garante colunas esperadas
for col, default in {
    "tema_normalizado": "",
    "macrotema": "",
    "subtema": "",
    "manter": True,
    "observacao": "",
    "qtd_ocorrencias": 0,
    "variantes_originais": "",
}.items():
    if col not in base_revisao.columns:
        base_revisao[col] = default

base_revisao["tema_normalizado"] = base_revisao["tema_normalizado"].apply(limpar_texto)
base_revisao = base_revisao[base_revisao["tema_normalizado"] != ""].drop_duplicates("tema_normalizado")
base_revisao = base_revisao.sort_values(["qtd_ocorrencias", "tema_normalizado"], ascending=[False, True]).reset_index(drop=True)

print(base_revisao.shape)
display(base_revisao.head(20))

## 5. Taxonomia de referência

Use esses macrotemas como padrão. O objetivo é criar hierarquia para o Power BI:

`macrotema → subtema → tema_normalizado`

In [ ]:
TAXONOMIA_REFERENCIA = {
    "Sustentabilidade e Clima": [
        "Clima", "Energia", "Meio Ambiente", "Biodiversidade", "Recursos Naturais", "Resiliência", "Descarbonização"
    ],
    "Economia e Produtividade": [
        "Crescimento Econômico", "Produtividade", "Indústria", "Serviços", "Agronegócio", "Trabalho e Renda", "Competitividade"
    ],
    "Tecnologia e Inovação": [
        "Tecnologias Digitais e Inovação", "Inteligência Artificial", "Dados", "Transformação Digital", "Pesquisa e Desenvolvimento"
    ],
    "Governança e Estado": [
        "Governança, Estado e Políticas Públicas", "Planejamento", "Regulação", "Capacidade Estatal", "Participação Social"
    ],
    "Inclusão e Sociedade": [
        "Inclusão, Equidade e Proteção Social", "Pobreza", "Desigualdade", "Gênero e Raça", "Demografia"
    ],
    "Infraestrutura e Território": [
        "Transportes e Logística", "Cidades e Urbanização", "Saneamento", "Desenvolvimento Regional", "Habitação"
    ],
    "Educação e Capacidades": [
        "Educação e Formação", "Capital Humano", "Qualificação Profissional", "Competências"
    ],
    "Saúde e Bem-estar": [
        "Saúde Pública", "Sistema de Saúde", "Bem-estar", "Saúde Mental", "Epidemias"
    ],
    "Segurança e Direitos": [
        "Segurança Pública", "Justiça", "Direitos Humanos", "Cidadania"
    ],
    "Agenda 2030 e ODS": [
        "Objetivos de Desenvolvimento Sustentável", "Agenda 2030", "Indicadores Globais", "Cooperação Internacional"
    ],
}

pd.DataFrame(
    [(macro, sub) for macro, subs in TAXONOMIA_REFERENCIA.items() for sub in subs],
    columns=["macrotema", "subtema"]
)

## 6. Regras automáticas de classificação

Estas regras só preenchem temas ainda vazios ou classificados como `Outros`; não sobrescrevem suas decisões manuais.

In [ ]:
regras = [
    {
        "padrao": r"óleo|diesel|gasolina|rodo|rodov|transp|aquav|ferrov|porto|portu|logíst|mobilidade|aviação|aero|hidrovia|frete|carga",
        "macrotema": "Infraestrutura e Território",
        "subtema": "Transportes e Logística",
    },
    {
        "padrao": r"clima|climát|carbono|emiss|descarbon|biodivers|ambient|floresta|energia|energét|renovável|recursos naturais|resiliên|adaptação|mitigação|sustentab|desmatamento|biocombust|biometano",
        "macrotema": "Sustentabilidade e Clima",
        "subtema": "Clima, Meio Ambiente e Energia",
    },
    {
        "padrao": r"educa|ensino|aprendiz|capacita|competên|qualifica|formação|escola|universidade|capital humano",
        "macrotema": "Educação e Capacidades",
        "subtema": "Educação e Formação",
    },
    {
        "padrao": r"saúde|sus|hospital|epidem|pandemia|vacina|mental|bem-estar|sanitár|doença|mortalidade|envelhecimento|covid|qualidade de vida",
        "macrotema": "Saúde e Bem-estar",
        "subtema": "Saúde Pública",
    },
    {
        "padrao": r"ia|inteligência artificial|digital|blockchain|dados|tecnolog|inova|internet|software|big data|automação|robótica|conectividade|ciber|telecomunica|transformação digital",
        "macrotema": "Tecnologia e Inovação",
        "subtema": "Tecnologias Digitais e Inovação",
    },
    {
        "padrao": r"pobreza|desigual|inclus|gênero|raça|racial|social|vulnerab|renda|equidade|proteção social|fome|segurança alimentar|juventude|idoso|criança|cultura",
        "macrotema": "Inclusão e Sociedade",
        "subtema": "Inclusão, Equidade e Proteção Social",
    },
    {
        "padrao": r"govern|estado|política pública|planejamento|regula|instituiç|gestão pública|participação|transparência|capacidade estatal|administração pública|federalismo|geopolítica|finança|corrupção|estratégic",
        "macrotema": "Governança e Estado",
        "subtema": "Governança, Estado e Políticas Públicas",
    },
    {
        "padrao": r"ods|agenda 2030|desenvolvimento sustentável|objetivos de desenvolvimento sustentável",
        "macrotema": "Agenda 2030 e ODS",
        "subtema": "Objetivos de Desenvolvimento Sustentável",
    },
    {
        "padrao": r"econom|produtiv|indústria|industrial|serviços|agronegócio|comércio|competitiv|mercado de trabalho|emprego|renda|investimento|fiscal|tribut|agricultura|empreendedor|mineração|gás",
        "macrotema": "Economia e Produtividade",
        "subtema": "Economia, Trabalho e Produtividade",
    },
    {
        "padrao": r"cidade|urbano|urbanização|habitação|moradia|saneamento|território|regional|região|infraestrutura|água|esgoto",
        "macrotema": "Infraestrutura e Território",
        "subtema": "Cidades, Saneamento e Desenvolvimento Regional",
    },
    {
        "padrao": r"segurança pública|violência|criminal|crime|justiça|direitos humanos|cidadania|sistema penal|prisional|defesa nacional",
        "macrotema": "Segurança e Direitos",
        "subtema": "Segurança, Justiça e Direitos",
    },
]

def aplicar_regras(base: pd.DataFrame, regras: list[dict]) -> pd.DataFrame:
    base = base.copy()
    for regra in regras:
        pendente = (
            base["macrotema"].isna()
            | (base["macrotema"].astype(str).str.strip() == "")
            | (base["macrotema"].astype(str).str.strip().str.lower() == "outros")
        )
        filtro = base["tema_normalizado"].str.contains(
            regra["padrao"], case=False, na=False, regex=True
        ) & pendente
        base.loc[filtro, "macrotema"] = regra["macrotema"]
        base.loc[filtro, "subtema"] = regra["subtema"]
    return base

base_revisao = aplicar_regras(base_revisao, regras)

print("Regras aplicadas.")
display(base_revisao.head(20))


## 7. Relatório de cobertura e pendências

In [ ]:
def calcular_pendencias(base: pd.DataFrame) -> pd.DataFrame:
    return base[
        base["macrotema"].isna()
        | (base["macrotema"].astype(str).str.strip() == "")
        | (base["macrotema"].astype(str).str.strip().str.lower() == "outros")
        | base["subtema"].isna()
        | (base["subtema"].astype(str).str.strip() == "")
        | (base["subtema"].astype(str).str.strip().str.lower() == "outros")
    ].copy()

pendencias = calcular_pendencias(base_revisao)

total = len(base_revisao)
pendentes = len(pendencias)
classificados = total - pendentes

print(f"Total de temas únicos: {total}")
print(f"Classificados: {classificados}")
print(f"Pendências restantes: {pendentes}")
print(f"Cobertura: {classificados / total:.1%}" if total else "Cobertura: n/a")

display(pendencias.sort_values("qtd_ocorrencias", ascending=False).head(50))

## 8. Célula para criar novas regras rapidamente

Use este padrão quando encontrar grupos de pendências semelhantes.

In [ ]:
# Exemplo de regra manual por filtro textual.
# Edite o padrão e os temas conforme necessário.

padrao = r"óleo|diesel|gasolina|rodo|transp|aquav"
macrotema = "Infraestrutura e Território"
subtema = "Transportes e Logística"

filtro = base_revisao["tema_normalizado"].str.contains(padrao, case=False, na=False, regex=True)

base_revisao.loc[filtro, "macrotema"] = macrotema
base_revisao.loc[filtro, "subtema"] = subtema

pendencias = calcular_pendencias(base_revisao)
print("Itens afetados:", filtro.sum())
print("Pendências restantes:", len(pendencias))
display(base_revisao.loc[filtro, ["tema_normalizado", "macrotema", "subtema", "qtd_ocorrencias"]].head(50))

## 9. Revisão manual direta de itens específicos

Use quando restarem poucos itens.

In [ ]:
# Exemplo:
# base_revisao.loc[base_revisao["tema_normalizado"].eq("Pobreza"), ["macrotema", "subtema"]] = [
#     "Inclusão e Sociedade", "Pobreza"
# ]

# Após editar, recalcule:
pendencias = calcular_pendencias(base_revisao)
print("Pendências restantes:", len(pendencias))

## 10. Gerar dimensão curada e salvar arquivos

Execute esta célula quando estiver satisfeito com a revisão.

In [ ]:
cols_dict = [
    "tema_normalizado",
    "macrotema",
    "subtema",
    "manter",
    "observacao",
    "qtd_ocorrencias",
    "variantes_originais",
]

# Garante ordem e existência das colunas
for col in cols_dict:
    if col not in base_revisao.columns:
        base_revisao[col] = "" if col != "manter" else True

pendencias = calcular_pendencias(base_revisao)

dim_tema_curada = base_revisao[
    ["tema_normalizado", "macrotema", "subtema", "manter", "observacao"]
].copy()

dim_tema_curada = dim_tema_curada[dim_tema_curada["manter"].astype(str).str.lower().isin(["true", "1", "sim", "yes"])]

base_revisao[cols_dict].to_csv(ARQ_DICIONARIO_TEMAS, index=False, encoding="utf-8-sig")
dim_tema_curada.to_csv(ARQ_DIM_TEMA_CURADA, index=False, encoding="utf-8-sig")
pendencias.to_csv(ARQ_PENDENCIAS_TEMAS, index=False, encoding="utf-8-sig")

relatorio = pd.DataFrame([
    {
        "total_temas_unicos": len(base_revisao),
        "classificados": len(base_revisao) - len(pendencias),
        "pendencias": len(pendencias),
        "cobertura": (len(base_revisao) - len(pendencias)) / len(base_revisao) if len(base_revisao) else 0,
    }
])
relatorio.to_csv(ARQ_RELATORIO_COBERTURA, index=False, encoding="utf-8-sig")

print("Dicionário salvo em:", ARQ_DICIONARIO_TEMAS)
print("Dimensão curada salva em:", ARQ_DIM_TEMA_CURADA)
print("Pendências salvas em:", ARQ_PENDENCIAS_TEMAS)
print("Relatório salvo em:", ARQ_RELATORIO_COBERTURA)
display(relatorio)

## 11. Conferência final

In [ ]:
print("Distribuição por macrotema:")
display(
    base_revisao.groupby("macrotema", dropna=False)
    .agg(qtd_temas=("tema_normalizado", "count"), ocorrencias=("qtd_ocorrencias", "sum"))
    .sort_values("ocorrencias", ascending=False)
)

print("Distribuição por macrotema e subtema:")
display(
    base_revisao.groupby(["macrotema", "subtema"], dropna=False)
    .agg(qtd_temas=("tema_normalizado", "count"), ocorrencias=("qtd_ocorrencias", "sum"))
    .sort_values("ocorrencias", ascending=False)
    .head(100)
)